# 在 Google Colab 上用免费 GPU 跑实验

这个笔记本（notebook）会：检查 GPU → 下载代码 → 安装依赖 → 跑一次完整实验 → 显示结果 → 打包下载结果。

**开始前**：菜单 **Runtime（代码执行程序）→ Change runtime type（更改运行时类型）→ 选 GPU（例如 T4）→ Save**。
然后从上到下，依次点每个格子（cell）左边的 ▶ 运行。

> 注意：Colab 的机器是临时的，关掉网页一段时间后里面的文件会被清空。跑完记得用最后一步把结果下载下来。

## 1. 检查 GPU
能看到显卡型号（例如 Tesla T4）就说明 GPU 可用；如果报错，回到上面把运行时类型改成 GPU。

In [ ]:
!nvidia-smi

## 2. 下载代码
从 GitHub 下载（clone）最新代码。重复运行这一格也没关系，会自动更新（pull）到最新版本。

In [ ]:
import os

if not os.path.exists("/content/KD"):
    !git clone https://github.com/jiangqu74-blip/KD.git /content/KD
%cd /content/KD
!git pull
!git log -1 --oneline

## 3. 安装依赖
Colab 已经装好了 GPU 版的 PyTorch，这一步只补装其余的包（package）。

In [ ]:
!pip install -q -r requirements.txt

## 4. 跑实验
想改参数：点左边的文件夹图标，打开 `KD/configs/default.yaml` 修改后保存，再运行这一格；
或者复制一份新的 yaml，把下面的 `configs/default.yaml` 换成新文件名。

In [ ]:
!python scripts/run_all.py --config configs/default.yaml --device cuda

## 5. 查看结果

In [ ]:
import glob

from IPython.display import Image, Markdown, display

latest = sorted(glob.glob("results/*/"))[-1]
print("最新结果文件夹:", latest)
report = open(latest + "REPORT.md", encoding="utf-8").read()
display(Markdown(report.replace("![Student comparison](student_ablation.png)", "")))
display(Image(latest + "student_ablation.png"))

## 6. 下载结果
下载 `results.zip`（包含所有结果文件夹）。

In [ ]:
!zip -qr /content/results.zip results
from google.colab import files

files.download("/content/results.zip")

## 7. 把结果放回 GitHub（二选一）

- **网页上传**：解压 `results.zip` → 在 GitHub 打开仓库的 `results` 文件夹 → **Add file → Upload files** →
  把新的结果文件夹拖进去 → **Commit changes**。
- **直接发给 Claude**：把结果文件夹里 `eval_results.json` 和 `REPORT.md` 的内容贴到云端会话里，让 Claude 分析。